In [2]:
import pandas as pd

# Load all 3 years
df_2022 = pd.read_csv("2022 OEP State-Level Public Use File.csv")
df_2023 = pd.read_csv("2023 OEP State-Level Public Use File.csv")
df_2024 = pd.read_csv("2024 OEP State-Level Public Use File.csv")

# Check size
print("2022 shape:", df_2022.shape)
print("2023 shape:", df_2023.shape)
print("2024 shape:", df_2024.shape)

# Peek at columns
df_2022.head()

2022 shape: (54, 101)
2023 shape: (54, 101)
2024 shape: (54, 102)


,State_Abrvtn,Pltfrm,Aplctn_Sbmtd,Indvdl_Aplctn_Sbmtd,QHP_Elgbl,FA_Elgbl,MC_Elgbl,Cnsmr,New_Cnsmr,Tot_Renrl,...,BHP_New_Enrl,BHP_Renrl,BHP_Age_0_17,BHP_Age_18_25,BHP_Age_26_34,BHP_Age_35_44,BHP_Age_45_54,BHP_Age_55,BHP_Male,BHP_Female
0,AK,HC.gov,"20,813","32,039","26,341","22,637","5,865","22,786","5,909","16,877",...,+,+,+,+,+,+,+,+,+,+
1,AL,HC.gov,"202,752","274,988","256,905","222,658","18,125","219,314","50,917","168,397",...,+,+,+,+,+,+,+,+,+,+
2,AR,HC.gov,"82,047","120,487","101,385","89,133","19,959","88,226","23,764","64,462",...,+,+,+,+,+,+,+,+,+,+
3,AZ,HC.gov,"166,660","258,899","233,857","199,673","26,874","199,706","48,940","150,766",...,+,+,+,+,+,+,+,+,+,+
4,CA,SBM,"3,434,646","6,763,131","2,511,840","2,020,949",NR,"1,777,442","255,575","1,521,867",...,+,+,+,+,+,+,+,+,+,+


In [3]:
# Keep only the columns we need
cols = ['State_Abrvtn', 'Cnsmr', 'New_Cnsmr', 'Tot_Renrl']

df_2022_clean = df_2022[cols].copy()
df_2023_clean = df_2023[cols].copy()
df_2024_clean = df_2024[cols].copy()

# Add a year column so we know which year each row belongs to
df_2022_clean['Year'] = 2022
df_2023_clean['Year'] = 2023
df_2024_clean['Year'] = 2024

# Stack all 3 years into one big table
df_all = pd.concat([df_2022_clean, df_2023_clean, df_2024_clean], ignore_index=True)

print("Total rows:", len(df_all))
df_all.head(10)

Total rows: 162


,State_Abrvtn,Cnsmr,New_Cnsmr,Tot_Renrl,Year
0,AK,"22,786","5,909","16,877",2022
1,AL,"219,314","50,917","168,397",2022
2,AR,"88,226","23,764","64,462",2022
3,AZ,"199,706","48,940","150,766",2022
4,CA,"1,777,442","255,575","1,521,867",2022
5,CO,"198,412","37,164","161,248",2022
6,CT,"112,633","19,267","93,366",2022
7,DC,"15,989","2,753","13,236",2022
8,DE,"32,113","7,087","25,026",2022
9,FL,"2,723,094","593,885","2,129,209",2022


In [4]:
print(df_all['Year'].value_counts())

Year
2022    54
2023    54
2024    54
Name: count, dtype: int64


In [5]:
# Remove commas from numbers and convert to integers
for col in ['Cnsmr', 'New_Cnsmr', 'Tot_Renrl']:
    df_all[col] = df_all[col].str.replace(',', '').str.strip()
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

# Calculate churn rate
df_all['Churn_Rate'] = ((df_all['Cnsmr'] - df_all['Tot_Renrl']) / df_all['Cnsmr'] * 100).round(2)

print(df_all.head(10))

  State_Abrvtn    Cnsmr  New_Cnsmr  Tot_Renrl  Year  Churn_Rate
0           AK    22786       5909      16877  2022       25.93
1           AL   219314      50917     168397  2022       23.22
2           AR    88226      23764      64462  2022       26.94
3           AZ   199706      48940     150766  2022       24.51
4           CA  1777442     255575    1521867  2022       14.38
5           CO   198412      37164     161248  2022       18.73
6           CT   112633      19267      93366  2022       17.11
7           DC    15989       2753      13236  2022       17.22
8           DE    32113       7087      25026  2022       22.07
9           FL  2723094     593885    2129209  2022       21.81


In [6]:
# Which states have the highest churn rate on average across all 3 years?
churn_by_state = df_all.groupby('State_Abrvtn')['Churn_Rate'].mean().round(2)
churn_by_state = churn_by_state.sort_values(ascending=False)
print(churn_by_state.head(15))

State_Abrvtn
WV    30.24
AR    29.05
TX    28.11
OH    27.95
MS    27.13
GA    26.97
AZ    26.83
LA    26.82
IN    26.52
TN    26.41
AL    25.45
SC    25.11
IL    24.49
AK    23.91
KS    23.87
Name: Churn_Rate, dtype: float64


In [7]:
# What's the national average churn rate?
national_avg = df_all['Churn_Rate'].mean().round(2)
print("National average churn rate:", national_avg, "%")

# How much higher are the top states vs national average?
print("\nTop states vs national average:")
for state in ['WV', 'AR', 'TX', 'MS']:
    state_avg = df_all[df_all['State_Abrvtn'] == state]['Churn_Rate'].mean().round(2)
    diff = (state_avg / national_avg).round(2)
    print(f"{state}: {state_avg}% — {diff}x the national average")

National average churn rate: 21.76 %

Top states vs national average:
WV: 30.24% — 1.39x the national average
AR: 29.05% — 1.34x the national average
TX: 28.11% — 1.29x the national average
MS: 27.13% — 1.25x the national average


In [8]:
#West Virginia's mid-year insurance churn rate is 1.4x the national average — and the top 15 highest-churn states are almost entirely Southern and rural.

In [9]:
# Let's see churn trend over the 3 years — is it getting worse or better?
churn_by_year = df_all.groupby('Year')['Churn_Rate'].mean().round(2)
print(churn_by_year)

Year
2022    20.65
2023    20.99
2024    23.64
Name: Churn_Rate, dtype: float64


In [ ]:
#That 2024 spike is interesting and tells a real story — this is likely because the extra subsidies from the American Rescue Plan started expiring, making insurance less affordable. That's a real business insight, not a made up one.
#So far we have found:

#National average churn is 21.76%
#southern/rural states churn up to 1.4x higher than average
#Churn is getting worse every year, with a big spike in 2024

In [10]:
# Save our clean data to a new CSV
df_all.to_csv("churn_clean.csv", index=False)
print("Saved! churn_clean.csv is ready.")

Saved! churn_clean.csv is ready.


In [11]:
# Load Census income data
df_income = pd.read_csv("ACSST1Y2022.S1901-Data.csv", skiprows=1)

# Peek at it
print(df_income.shape)
df_income.head(3)

(52, 131)


,Geography,Geographic Area Name,Estimate!!Households!!Total,Margin of Error!!Households!!Total,"Estimate!!Households!!Total!!Less than $10,000","Margin of Error!!Households!!Total!!Less than $10,000","Estimate!!Households!!Total!!$10,000 to $14,999","Margin of Error!!Households!!Total!!$10,000 to $14,999","Estimate!!Households!!Total!!$15,000 to $24,999","Margin of Error!!Households!!Total!!$15,000 to $24,999",...,Margin of Error!!Nonfamily households!!Median income (dollars),Estimate!!Nonfamily households!!Mean income (dollars),Margin of Error!!Nonfamily households!!Mean income (dollars),Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Household income in the past 12 months,Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Household income in the past 12 months,Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Family income in the past 12 months,Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Family income in the past 12 months,Estimate!!Nonfamily households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months,Margin of Error!!Nonfamily households!!PERCENT ALLOCATED!!Nonfamily income in the past 12 months,Unnamed: 130
0,0400000US01,Alabama,2016448,11475,7.6,0.3,5.2,0.3,9.3,0.3,...,919,47145,1778,(X),(X),(X),(X),34.2,(X),NaN
1,0400000US02,Alaska,274574,3261,4.4,0.7,2.5,0.4,5.4,0.8,...,5211,74242,4733,(X),(X),(X),(X),30.3,(X),NaN
2,0400000US04,Arizona,2850377,11519,5.4,0.3,3.0,0.2,6.2,0.3,...,1067,66724,1965,(X),(X),(X),(X),33.6,(X),NaN


In [12]:
# Find the median income column
median_cols = [col for col in df_income.columns if 'Median' in col and 'Households' in col]
print(median_cols)

['Estimate!!Households!!Median income (dollars)', 'Margin of Error!!Households!!Median income (dollars)']


In [13]:
# Keep only state name and median income
df_income_clean = df_income[['Geographic Area Name', 'Estimate!!Households!!Median income (dollars)']].copy()

# Rename columns to something simpler
df_income_clean.columns = ['State_Name', 'Median_Income']

# Preview
print(df_income_clean.shape)
df_income_clean.head(5)

(52, 2)


,State_Name,Median_Income
0,Alabama,59674
1,Alaska,88121
2,Arizona,74568
3,Arkansas,55432
4,California,91551


In [14]:
# Create a mapping of state name to abbreviation
state_map = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR',
    'California': 'CA', 'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE',
    'District of Columbia': 'DC', 'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI',
    'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS',
    'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV',
    'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',
    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY'
}

# Add abbreviation column
df_income_clean['State_Abrvtn'] = df_income_clean['State_Name'].map(state_map)

# Drop rows where we couldn't match (territories etc.)
df_income_clean = df_income_clean.dropna(subset=['State_Abrvtn'])

print(df_income_clean.shape)
df_income_clean.head(5)

(51, 3)


,State_Name,Median_Income,State_Abrvtn
0,Alabama,59674,AL
1,Alaska,88121,AK
2,Arizona,74568,AZ
3,Arkansas,55432,AR
4,California,91551,CA


In [15]:
# Join churn data with income data
df_merged = df_all.merge(df_income_clean[['State_Abrvtn', 'Median_Income']], 
                          on='State_Abrvtn', 
                          how='left')

print(df_merged.shape)
df_merged.head(5)

(162, 7)


,State_Abrvtn,Cnsmr,New_Cnsmr,Tot_Renrl,Year,Churn_Rate,Median_Income
0,AK,22786,5909,16877,2022,25.93,88121.0
1,AL,219314,50917,168397,2022,23.22,59674.0
2,AR,88226,23764,64462,2022,26.94,55432.0
3,AZ,199706,48940,150766,2022,24.51,74568.0
4,CA,1777442,255575,1521867,2022,14.38,91551.0


In [16]:
# Save final merged dataset
df_merged.to_csv("churn_final.csv", index=False)
print("Done! churn_final.csv is ready.")
print(f"Total rows: {len(df_merged)}")
print(f"Columns: {list(df_merged.columns)}")

Done! churn_final.csv is ready.
Total rows: 162
Columns: ['State_Abrvtn', 'Cnsmr', 'New_Cnsmr', 'Tot_Renrl', 'Year', 'Churn_Rate', 'Median_Income']


In [25]:
import snowflake.connector

conn = snowflake.connector.connect(
    user='HARSHITA',
    password='YOUR_PASSWORD_HERE',
    account='rbvusrq-ei40252'
)

print("Connected to Snowflake successfully!")

DatabaseError: 250001 (08001): Failed to connect to DB: rbvusrq-ei40252.snowflakecomputing.com:443. Incorrect username or password was specified.

In [19]:
cursor = conn.cursor()

# Create database and table
cursor.execute("CREATE DATABASE IF NOT EXISTS INSURANCE_CHURN")
cursor.execute("USE DATABASE INSURANCE_CHURN")
cursor.execute("CREATE SCHEMA IF NOT EXISTS ANALYTICS")
cursor.execute("USE SCHEMA ANALYTICS")

cursor.execute("""
    CREATE OR REPLACE TABLE CHURN_FINAL (
        STATE_ABRVTN VARCHAR(5),
        CNSMR NUMBER,
        NEW_CNSMR NUMBER,
        TOT_RENRL NUMBER,
        YEAR NUMBER,
        CHURN_RATE FLOAT,
        MEDIAN_INCOME FLOAT
    )
""")

print("Database and table created!")

Database and table created!


In [21]:
    # Rename columns to uppercase to match Snowflake table
    df_upload = df_merged.copy()
    df_upload.columns = [col.upper() for col in df_upload.columns]
    print(df_upload.columns.tolist())

    # Upload to Snowflake
    success, nchunks, nrows, _ = write_pandas(conn, df_upload, 'CHURN_FINAL')

    print(f"Upload successful: {success}")
    print(f"Rows uploaded: {nrows}")

['STATE_ABRVTN', 'CNSMR', 'NEW_CNSMR', 'TOT_RENRL', 'YEAR', 'CHURN_RATE', 'MEDIAN_INCOME']
Upload successful: True
Rows uploaded: 162


In [23]:
cursor.execute("SELECT * FROM CHURN_FINAL LIMIT 5")
results = cursor.fetchall()
for row in results:
    print(row)

('AK', 22786, 5909, 16877, 2022, 25.93, 88121.0)
('AL', 219314, 50917, 168397, 2022, 23.22, 59674.0)
('AR', 88226, 23764, 64462, 2022, 26.94, 55432.0)
('AZ', 199706, 48940, 150766, 2022, 24.51, 74568.0)
('CA', 1777442, 255575, 1521867, 2022, 14.38, 91551.0)


In [24]:
cursor.execute("""
    CREATE OR REPLACE VIEW CHURN_SUMMARY AS
    SELECT 
        STATE_ABRVTN,
        ROUND(AVG(CHURN_RATE), 2) AS AVG_CHURN_RATE,
        ROUND(AVG(MEDIAN_INCOME), 0) AS AVG_MEDIAN_INCOME,
        COUNT(*) AS YEARS_OF_DATA
    FROM CHURN_FINAL
    GROUP BY STATE_ABRVTN
    ORDER BY AVG_CHURN_RATE DESC
""")
print("View created!")

View created!
